## FINAL 

## A. INTRODUCTION

*a.	It is recommended that you complete the introduction after you have finished all the other parts.*

*b.	State whether you solved your project as a regression or classification problem.*
I solved as Classification


*c.	The general proposal comments including instructions for how to convert a regression problem to a classification problem if that is what you are interested in working on.*

* Converting this problem to classification required creating a a new column for binary classification from `popularity`, with 1 = >50, and 0 =<50


*d.	Describe the major findings you have identified based on your analysis.*

* Major findings
    * This dataset was not as clear-cut as the samples used during class, this presented challenges.
    * Although the end model was not as accurate as hoped for, it still produced a greater accuracy than the intercept only model



*e.	Which inputs/features seem to influence the response/outcome the most?*

* The inputs/features that influence the outcome the most are:
    * playlist_subgenre
    * Loudness
    * Energy

      
*f.	What supported your conclusion? Was it only through predictive models?*

* the predictive models mainly supported this conclustion, but also the EDA with categorical and classifiacation 


*g.	Could EDA help identify similar trends/relationships?*

* yes but to a lesser degree


*h.	Was clustering consistent with any conclusions from the predictive models?*

* I did not have much sucess with drawing the same conclusions from clustering this dataset with modeling


*i.	What skills did you learn from going through this project?*

* Many skills, especially the need for data manipulation to set up data





In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import seaborn as sns


In [ ]:
songs_url = 'https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-01-21/spotify_songs.csv'

In [ ]:
df = pd.read_csv( songs_url ) 

---

## B. EDA

### EDA - Basic Information

#### 1. *Number of rows and columns*

In [ ]:
df.shape

#### 2. *Variable Names*
#### 3. *Data Types*

In [ ]:
df.info()

`track_id` does NOT contain disctinct variables. I will not group or clean duplicate records at this time as this was NOT part of project instructions 

#### 4. *Number of mising values per variable*

In [ ]:
df.isna().sum()

#### 5. *Number of unique values per variable*

In [ ]:
df.nunique()

*Data Cleanup*

I will drop the duplicates to aid in exploration and reduce the variables. I will keep the first appearence of `track_id` from the data

I will drop missing as this may impact clustering later. Data set is large and the 5 missings will not impact analysis. 

see file "...EDA_duplicate_analysis"

In [ ]:
df_nodupes = df.drop_duplicates(
    subset=['track_id'], 
    keep='first'
)

In [ ]:
df_clean = df_nodupes.dropna()

*Notes on Categorical / Countinuous variables:*

* MODE has 2 values but this does NOT indicated a 'yes' or 'no', or 'true'/'false' in this case. Per the data dictionary "Mode indicates the modality (major or minor) of a track"
* KEY is a musical key. Definition: Integers map to pitches using standard Pitch Class notation . E.g. 0 = C, 1 = C♯/D♭, 2 = D,
* Data Dictionary referenced https://github.com/rfordatascience/tidytuesday/tree/master/data/2020/2020-01-21I
* I will treat both of these columns as non-numeric columns. I will convert only 'mode' to Object though as I may want to vizualise `key` in countinuous relationship during explortation

I will add the `binary_outcome` for `track_popularity` to use in a 'classification' problem

In [ ]:
df_clean.loc[:, 'mode'] = df_clean['mode'].astype(str)

`binary_outcome` will be used as event / non-event classification. Event = '1', non-event = '0'. The ratio is not even, I will review further in modeling. 

In [ ]:
df_clean['binary_outcome'] = np.where ( df_clean.track_popularity >= 50, 1, 0)

In [ ]:
df_clean.binary_outcome.value_counts(normalize=True)

Notes on below code:
* Plots returned errors with track and album titles containing '$' charactors. Below code replaces with the the escaped version
* Plots also returned many errors due to font issues (foreign languages) in track and album titles.
* Reference Source: Gemini

In [ ]:
df_clean.loc[:, 'track_name'] = df_clean['track_name'].str.replace('$', r'\$', regex=False)
df_clean.loc[:, 'track_artist'] = df_clean['track_artist'].str.replace('$', r'\$', regex=False)
df_clean.loc[:, 'track_album_name'] = df_clean['track_album_name'].str.replace('$', r'\$', regex=False)
df_clean.loc[:, 'playlist_name'] = df_clean['playlist_name'].str.replace('$', r'\$', regex=False)

plt.rcParams['font.family'] = [
    'Arial',          # Excellent general Unicode support (good for Hebrew/Basic Arabic)
    'Segoe UI',      # Windows default, often has good coverage
    'Yu Gothic',      # Japanese/CJK support (for Katakana)
    'Malgun Gothic',  # Korean/Hangul support 
    'sans-serif'      # Fallback to a generic sans-serif 
]


plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.default'] = 'regular'


Since the goal is to predict if a song is popular or not, I will add a feature display the count of each song if repeated with unique track_ids by combining track name and artist name

In [ ]:
df_clean['track_and_artist'] = df_clean['track_name'] + ' - ' + df_clean['track_artist']

In [ ]:
df_clean.nunique()

In [ ]:
track_artist_vol=df_clean.track_and_artist.value_counts()

In [ ]:
track_artist_vol

In [ ]:
df_clean = df_clean.merge(track_artist_vol, on='track_and_artist', how='left')

### EDA - Visualization

#### 1. *Counts of categorical variables*

In [ ]:
fig, ax = plt.subplots(figsize = (12,4))

df_clean.track_id.value_counts(dropna = False).head(10).plot (kind = 'bar', ax=ax)
plt.show()

`track_id` is distinct

In [ ]:
fig, ax = plt.subplots(figsize = (12,4))

df_clean.track_name.value_counts(dropna = False).head(10).plot (kind = 'bar', ax=ax)
plt.show()

Duplicate `track_names` may be generic names. `track_id`s may have the same name in data.

In [ ]:
fig, ax = plt.subplots(figsize = (12,4))

df_clean.track_artist.value_counts(dropna = False).head(10).plot (kind = 'bar', ax=ax)

plt.show()

`track_artist` may appear more than once. An artist may have many songs. 

In [ ]:
fig, ax = plt.subplots(figsize = (12,4))

df_clean.track_album_name.value_counts(dropna = False).head(10).plot (kind = 'bar', ax=ax)

plt.show()

`track_album_id` (not visualized) and `track_album_name` may have more than one value

In [ ]:
fig, ax = plt.subplots(figsize = (12,4))

df_clean.track_album_release_date.value_counts(dropna = False).head(10).plot (kind = 'bar', ax=ax)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize = (12,4))

df_clean.playlist_name.value_counts(dropna = False).head(10).plot (kind = 'bar', ax=ax)

plt.show()

`playlist_id` (not visualized) and `playlist_name` may have more than one value

In [ ]:
fig, ax = plt.subplots(figsize = (12,4))

df_clean.playlist_genre.value_counts(dropna = False).plot (kind = 'barh', ax=ax)

plt.show()

`playlist_genre` has 6 possible values, and not any genre dominates the data

In [ ]:
fig, ax = plt.subplots(figsize = (12,4))

df_clean.playlist_subgenre.value_counts(dropna = False).plot (kind = 'barh', ax=ax)

plt.show()

a `playlist_subgenre` may have more than double the count as another sub_genre

In [ ]:
fig, ax = plt.subplots(figsize = (12,4))

df_clean['mode'].value_counts(dropna = False).plot (kind = 'barh', ax=ax)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize = (12,4))

df_clean.key.value_counts(dropna = False).plot (kind = 'barh', ax=ax)

plt.show()

In order to analyize date values more simply, I will convert `track_album_release_date` to `release_year`. Assistance with argments sourced from 'feedback' file.

In [ ]:
df_clean.loc[:, 'release_year'] = df_clean['track_album_release_date'].str[:4].astype(int)

In [ ]:
fig, ax = plt.subplots(figsize = (12,4))

df_clean.release_year.value_counts(dropna = False).plot (kind = 'bar', ax=ax)

plt.show()

#### 2.	*Distributions of continuous variables*

In [ ]:
sns.displot(data = df_clean, x = 'track_popularity', kind = 'hist', kde=True)

plt.show()

`track_popularity` has abnormal distrubtion due to many records with zero or near zero popularity

In [ ]:
sns.displot(data = df_clean, x = 'danceability', kind = 'hist', kde=True)

plt.show()

`dancibility` gaussian-like distrubtion, slightly right skewed.

In [ ]:
sns.displot(data = df_clean, x = 'energy', kind = 'hist', kde=True)

plt.show()

`energy` is right skewed

In [ ]:
sns.displot(data = df_clean, x = 'loudness', kind = 'hist', kde=True)

plt.show()

`loudness` is not normal, right skewed

In [ ]:
sns.displot(data = df_clean, x = 'speechiness', kind = 'hist', kde=True)

plt.show()

`speechiness` is not normal distribution,  heavily left skewed

In [ ]:
sns.displot(data = df_clean, x = 'acousticness', kind = 'hist', kde=True)

plt.show()

`acoustic` is not normal distribution

In [ ]:
sns.displot(data = df_clean, x = 'instrumentalness', kind = 'hist', kde=True)

plt.show()

`instrumentalness` is not normal distirbution, many values at zero or near zero

In [ ]:
sns.displot(data = df_clean, x = 'liveness', kind = 'hist', kde=True)

plt.show()

`liveness` is not normal distribution

In [ ]:
sns.displot(data = df_clean, x = 'valence', kind = 'hist', kde=True)

plt.show()

`valence` is not normal, without tails

In [ ]:
sns.displot(data = df_clean, x = 'tempo', kind = 'hist', kde=True)

plt.show()

`tempo` is not normal distribution 

In [ ]:
sns.displot(data = df_clean, x = 'duration_ms', kind = 'hist', kde=True)

plt.show()

`duration_ms` is gaussian like 

#### 3.	*Relationships between continuous variables*

I will reshape data prior now to long format to facilitate Seaborn charts more easily.

In [ ]:
df_clean_for_lf = df_clean.copy()

In [ ]:
df_clean_for_lf.loc[:, 'binary_outcome'] = df_clean_for_lf['binary_outcome'].astype('str')

In [ ]:
df_clean_for_lf.info()

In [ ]:
data_features = df_clean_for_lf.select_dtypes('number').copy()

In [ ]:
data_features.melt(ignore_index = False).loc[0]

In [ ]:
lf = data_features.reset_index().\
rename(columns={'index': 'rowid'}).\
melt(id_vars=['rowid'])

In [ ]:
lf.loc[lf.rowid ==0,:]

In [ ]:
data_objects= df_clean_for_lf.select_dtypes('object').copy()

In [ ]:
data_objects.info()

In [ ]:
id_cols = [ 'rowid'] + data_objects.columns.to_list()

In [ ]:
data_lf= df_clean_for_lf.reset_index().\
rename(columns={'index':'rowid'}).\
melt(id_vars=id_cols,value_vars=data_features.columns)

In [ ]:
data_lf.info()

In [ ]:
sns.displot(data= data_lf, x ='value', col='variable', kind ='kde', 
            hue ='binary_outcome',  row ='playlist_genre', 
           facet_kws={'sharex':False, 'sharey':False},
           common_norm=False)

plt.show() 

In [ ]:
sns.displot(data= data_lf, x ='value', col='variable', kind ='kde', 
            hue ='playlist_genre',  col_wrap=4,
           facet_kws={'sharex':False, 'sharey':False},
           common_norm=False)

plt.show() 

In [ ]:
sns.catplot(data=data_lf, x='playlist_genre', y='value', col='variable', col_wrap=4,
            kind='box', hue='playlist_genre',
            sharey=False, sharex=False)
plt.show()

In [ ]:
sns.catplot(data= df_clean, x='playlist_genre', y='energy', kind='violin', aspect=2)

plt.show()

In [ ]:
sns.catplot(data= df_clean, x='playlist_genre', y='danceability', kind='violin', aspect=2)

plt.show()

From Above, there is a difference in conditional distribution for `dancibility` when grouped by `playlist_genre`. "rock" has a lower innerquartile range then other genres. 

Also, there is a conditional difference in distribution for `energy` when grouped by `playlist_genre`. "edm" has a higher innerquartile range then "r&b" with no overlap. 


In [ ]:
sns.catplot(data = data_lf, x='playlist_genre', y='value', col='variable',hue='binary_outcome', col_wrap=4, kind = 'point', aspect=1, linestyle= 'none',
           sharey=False, sharex=False)

plt.show()

The point plot shows there are differneces in confidence intravales for most variables when grouped by `playlist_genre`, as well as `binary_outcome`. This is interesting to know if we were to prediect `binary_outcome`.

In [ ]:
fig, ax = plt.subplots()

sns.heatmap(data= df_clean.corr(numeric_only=True),
            vmin=-1, vmax=1, center=0,
            cmap='coolwarm',
            cbar=False,
            annot=True, annot_kws={'size': 5},
            ax=ax)
plt.show()

Positive coorelation between `energy` and `loudness`. Negative coorelation between `energy` and `accousticness`

In [ ]:
sns.pairplot(data=df_clean, hue='key', 
            diag_kws= {'common_norm': False}, 
            plot_kws={'alpha': 0.3, 's': 5}) # Due to number of observations, I am adding transparency and reducing the point size. (Source Ref: Gemini))

plt.show()

In [ ]:
sns.pairplot(data=df_clean, hue='mode', 
            diag_kws= {'common_norm': False}, 
            plot_kws={'alpha': 0.3, 's': 5}) # Due to number of observations, I am adding transparency and reducing the point size. (Source Ref: Gemini))

plt.show()

In [ ]:
sns.pairplot(data=df_clean, hue='playlist_genre', 
            diag_kws= {'common_norm': False}, 
            plot_kws={'alpha': 0.5, 's': 5}) # Due to number of observations, I am adding transparency and reducing the point size. (Source Ref: Gemini))

plt.show()

Countinuous to Continuos analysis. I do not find anything that jumps out at me from above analysis, other than the Positive coorelation between `energy` and `loudness`. Negative coorelation between `energy` and `accousticness`. There may be too many data points to compare via scatter plot. 

In [ ]:
sns.pairplot(data=df_clean, hue='binary_outcome', 
            diag_kws= {'common_norm': False}, 
            plot_kws={'alpha': 0.3, 's': 5}) # Due to number of observations, I am adding transparency and reducing the point size. (Source Ref: Gemini))

plt.show()

From above pairs plot, it shows conditional distribution and relationship between continuous inputs but there are no distinguishable patterns in relation to outcome I can see, potentially due to size of sample.

#### iii. Visualize the counts of combinations between the response (outcome) and categorical inputs.

In [ ]:
sns.catplot(data = df_clean, x = 'binary_outcome', hue ='playlist_genre', kind = 'count')

plt.show()

For `binary_outcome` of "high" the most prevelent observations are from `playlist_genre` "pop". "edm" has the lowest popularity ratio amongst `playlist_genre`.

In [ ]:
sns.catplot(data = df_clean, x = 'count', hue ='binary_outcome', kind = 'count')

plt.show()

There is a higher ratio of `binary_outcome` '1' for tracks that are repeated on dataset

In [ ]:
sns.catplot( data = df_clean, x='playlist_subgenre', col= 'playlist_genre', hue='binary_outcome', kind='count', col_wrap = 3, sharex=False, sharey = False, aspect = 1 )

plt.show()

On the above chart we can easily visualize which `sub_genre` are the more popular than not. 'dance pop', 'post-teen pop', 'hip hop', 'trap', 'permanent wave' 'reggaeton', and 'urban contemporary' are all more popular than not. Using genre or sub_genre may be a good indicator of whether or not a song is popular. 

In [ ]:
sns.catplot( data = df_clean, x='mode', hue='binary_outcome', kind='count', sharex=False, sharey = False, aspect = 1 )

plt.show()

For `binary_outcome` by `mode` the counts look to be influeced by obvservation count, does not seem interesting

In [ ]:
sns.catplot( data = df_clean, x='key', hue='binary_outcome', kind='count', sharex=False, sharey = False, aspect = 2 )

plt.show()

For `binary_outcome` by `key` the counts look to be influeced by obvservation count, does not seem interesting

## C.	Cluster Analysis. 

1. I am using 12 input variables, as these are the countinious/number variables. I am using all 32828 observations.
2. Most the variables are NOT gaussian-like. `duration_ms` is gaussian -like
3. The variables are NOT highly coorelated. I display the correlation coefficient in previuos section. There was a positive correlation between `energy` and `loudness`, and a negative coorrelation between `energy` and `acousticness`.
4. I dropped missing values, this was a very small amount compared to total observations.

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
data_standardize = StandardScaler()

In [ ]:
Xdata = StandardScaler().fit_transform(data_features)

In [ ]:
sns.catplot(data=pd.DataFrame(Xdata, columns=data_features.columns), kind='box', aspect=2)

plt.show()

Checking if standardizedd, There are some extremes but the data is more scaled than previously. 

In [ ]:
from sklearn.cluster import KMeans

In [ ]:
cluster_2 = KMeans(n_clusters=2, random_state=121, n_init = 25, max_iter=500).fit_predict(Xdata)

In [ ]:
df_clean_copy = df_clean.copy()

In [ ]:
df_clean_copy['k2'] = pd.Series(cluster_2, index=df_clean_copy.index).astype('category')

In [ ]:
sns.relplot(data= df_clean_copy, x='acousticness', y='energy', hue='k2')

plt.show()

In [ ]:
sns.relplot(data= df_clean, x='acousticness', y='energy', hue='binary_outcome')

plt.show()

In [ ]:
sns.catplot(data=df_clean_copy, x='playlist_genre', hue='k2', kind='count')

plt.show()

Use 2 clusters does not seem to provide a definitive result. I will look for optimal n cluster 

In [ ]:
tots_within = []

K= range(1,31)

for k in K:
    km = KMeans(n_clusters = k, random_state=121, n_init=25, max_iter=500)
    km=km.fit(Xdata)

    tots_within.append(km.inertia_ )

In [ ]:
fig, ax=plt.subplots()

ax.plot(K, tots_within, 'bo-')
ax.set_xlabel('number of cluster')
ax.set_ylabel('ntotal within sume of squares')

plt.show()

There is no definitive elbow, I will try 6 clusters `playlist_genre` and 12 clusters `playlist_subgenre` to see if results are interesting. 

In [ ]:
cluster_6 = KMeans(n_clusters=6, random_state=121, n_init = 25, max_iter=500).fit_predict(Xdata)

In [ ]:
df_clean_copy['k6'] = pd.Series(cluster_6, index=df_clean_copy.index).astype('category')

In [ ]:
cluster_12 = KMeans(n_clusters=12, random_state=121, n_init = 25, max_iter=500).fit_predict(Xdata)

In [ ]:
df_clean_copy['k12'] = pd.Series(cluster_12, index=df_clean_copy.index).astype('category')

In [ ]:
sns.relplot(data= df_clean_copy, x='acousticness', y='energy', hue='k6')

plt.show()

In [ ]:
sns.relplot(data= df_clean_copy, x='loudness', y='energy', hue='k6')

plt.show()

In [ ]:
sns.relplot(data= df_clean_copy, x='acousticness', y='energy', hue='k12')

plt.show()

In [ ]:
sns.relplot(data= df_clean_copy, x='loudness', y='energy', hue='k12')

plt.show()

1. The clusters do not align to known groupings, possibly due to the high number of observations.
2.  NO, when using 2 clusters for 2 continuous variables, as shown above, the clusters do not align with `binary_outcome`
3.  The Conditional distribution of the variables I used are the clustering groups derived by KMeans, first 2 then 6 and 12. 

In [ ]:
df_clean_copy.info()

In [ ]:
sns.catplot(data=df_clean_copy, x='binary_outcome', hue='k6', kind='count')

plt.show()

## D. Models: Fitting and Interpretation

---

In [ ]:
df_clean.info()

In [ ]:
train_data=df_clean[['playlist_genre', 'playlist_subgenre', 'key', 'mode', 'danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'binary_outcome', 'release_year', 'count']].copy()

In [ ]:
train_data.info()


In [ ]:
train_data.head(10)

### Preprocessing

I will transfrom continuous variables that were not guassian. Becuase this includes both positive and negative numbers, I am using yeojohnson method #source gemini

I will then standarize for scale 

In [ ]:
from sklearn.preprocessing import PowerTransformer

In [ ]:
# 1. Identify the columns to transform
yeojohnson_cols = ['loudness', 'acousticness', 'speechiness', 'instrumentalness', 'liveness','duration_ms', 'release_year', 'tempo']
df_to_transform = train_data[yeojohnson_cols]

In [ ]:
pt = PowerTransformer(method='yeo-johnson') #source gemini

In [ ]:
transformed_array = pt.fit_transform(df_to_transform)

In [ ]:
df_transformed = pd.DataFrame(
    transformed_array,
    columns=yeojohnson_cols,
    index=df_to_transform.index
)

In [ ]:
# 5. Drop the original columns from the main DataFrame and rejoin the transformed ones
train_data_pt = train_data.drop(columns=yeojohnson_cols)
train_data_pt = pd.concat([train_data_pt, df_transformed], axis=1)

In [ ]:
train_data_pt.info()

In [ ]:
train_data_pt.loc[:, 'key'] = train_data_pt['key'].astype(str)

In [ ]:
train_data_pt.loc[:, 'binary_outcome'] = train_data_pt['binary_outcome'].astype(str)

In [ ]:
train_data_features = train_data_pt.select_dtypes('number').copy()

In [ ]:
train_data_features_df =pd.DataFrame(train_data_features,columns=train_data_features.columns, index=train_data_features.index)

In [ ]:
sns.catplot(data=train_data_features_df, kind='box', aspect=2)

plt.show()

In [ ]:
df_categorical = train_data[['playlist_genre', 'playlist_subgenre', 'key', 'mode','binary_outcome']]

In [ ]:
train_data_combine = pd.concat([train_data_features_df, df_categorical], axis=1)

In [ ]:
train_data_combine.info()

In [ ]:
train_data_combine.describe()

In [ ]:
import statsmodels.formula.api as smf

from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score

In [ ]:
def fit_and_assess_logistic (mod_name, a_formula, train_data, threshold):
    a_mod = smf.logit( formula= a_formula, data=train_data).fit()

    train_copy = train_data.copy()

    train_copy['pred_probability'] = a_mod.predict(train_data)

    train_copy['pred_class'] = np.where (train_copy.pred_probability > threshold, 1, 0 )

    TN, FP, FN, TP = confusion_matrix( train_copy.binary_outcome.to_numpy(), train_copy.pred_class.to_numpy() ).ravel()

    Accuracy = (TN + TP) / (TN + FP + FN + TP)

    Sensitivity = (TP) / (TP + FN)

    Specificity = (TN) / (TN + FP)

    FPR = 1 - Specificity

    ROC_AUC = roc_auc_score ( train_copy.binary_outcome.to_numpy(), train_copy.pred_probability.to_numpy() )

    res_dict = {'model_name': mod_name, 
                'model_formula': a_formula,
                'num_coefs': len( a_mod.params), 
                'threshold': threshold,
                'Accuracy': Accuracy,
                'Sensitivity': Sensitivity,
                'Specificity': Specificity,
                'FPR': FPR, 
                'ROC_AUC': ROC_AUC}

    return pd.DataFrame (res_dict, index=[0])



In [ ]:
def my_coefplot(mod, figsize_use=(10,4)):
    fig, ax= plt.subplots(figsize=figsize_use)

    ax.errorbar( y=mod.params.index,
                x = mod.params,
                xerr = 2 * mod.bse,
                fmt= 'o', color ='k', ecolor = 'k', elinewidth = 2, ms=10)

    ax.axvline( x=0, linestyle='--', linewidth = 3.5, color='grey')

    ax.set_xlabel('coefficient value')

    plt.show()

##### Formula List

I will run each model seperately due to max iterations errors with the created function. 

Model_8 is the model with the most coefficients that ran effectively. 

Model_9  I will choose features that seemed intereting from EDA to reduce formula length /coefficient count. I am removing the derived columns `release_year` and `count`. Also removing `duration_ms`

Model_10 -Model_12 I will remove same as model 9 plus `valence` and `mode` and `playlist_genre` (keeping sub_genre)

In [ ]:
model_1 = 'binary_outcome ~ 1' # model 1 - Intercept only

In [ ]:
model_2 = 'binary_outcome ~ playlist_genre' # model 2

In [ ]:
model_3 = 'binary_outcome ~ playlist_subgenre' # model 3

In [ ]:
model_4 = 'binary_outcome ~ key' # model 4

In [ ]:
model_5 = 'binary_outcome ~ mode' # model 5

In [ ]:
model_6 = 'binary_outcome ~ mode + playlist_genre + playlist_subgenre + key + mode' # model 6- Categorical inputs with additive features
                # model 6- Categorical inputs with additive features

In [ ]:
model_7 = 'binary_outcome ~ danceability + energy+ loudness + speechiness + acousticness+ instrumentalness + liveness + valence + tempo + duration_ms + release_year + count'
                # model 7 - Continuous inputs with linear additive features.

In [ ]:
model_8 = 'binary_outcome ~ danceability + energy+ loudness + speechiness + acousticness+ instrumentalness + liveness + valence + tempo + duration_ms + release_year + playlist_genre + playlist_subgenre + key + mode + count'
                # model 8 - All inputs (continuous and categorical) with linear additive features.

In [ ]:
model_9 = 'binary_outcome ~ danceability * energy* loudness * speechiness * acousticness* instrumentalness * liveness * valence * tempo'
                # model 9 - Continuous inputs with linear main effect and pair-wise interactions.

In [ ]:
model_10 ='binary_outcome ~ (danceability + energy+ loudness + speechiness + acousticness + instrumentalness + liveness + tempo) * playlist_subgenre * key'
                # model 10 - Interact the categorical inputs with the continuous inputs. This model must include the linear main effects as well (the formula interface helps with the interactions).
               

In [ ]:
model_11 = 'binary_outcome ~ danceability + energy + loudness + speechiness + acousticness + instrumentalness + liveness + tempo + np.power(danceability,2) + np.power(energy,2) + np.power(loudness, 2) + np.power(speechiness,2) + np.power(acousticness,2) + np.power(instrumentalness,2) + np.power(liveness, 2) + np.power(tempo,2)'
                # model 11 extra

In [ ]:
model_12 = 'binary_outcome ~ playlist_subgenre + key + danceability + energy + loudness + speechiness + acousticness + instrumentalness + liveness + tempo  + np.power(danceability,2) + np.power(energy,2) + np.power(loudness, 2) + np.power(speechiness,2) + np.power(acousticness,2) + np.power(instrumentalness,2) + np.power(liveness, 2) + np.power(tempo,2)'
                # model 12extra

In [ ]:
results_list = []

results_list.append( fit_and_assess_logistic('model_1', model_1, train_data=train_data_combine, threshold= 0.5))

In [ ]:
model_1_results_df= pd.concat( results_list, ignore_index = True)

In [ ]:
results_list = []

results_list.append( fit_and_assess_logistic('model_2', model_2, train_data=train_data_combine, threshold= 0.5))

In [ ]:
model_2_results_df= pd.concat( results_list, ignore_index = True)

In [ ]:
results_list = []

results_list.append( fit_and_assess_logistic('model_3', model_3, train_data=train_data_combine, threshold= 0.5))

In [ ]:
model_3_results_df= pd.concat( results_list, ignore_index = True)

In [ ]:
results_list = []

results_list.append( fit_and_assess_logistic('model_4', model_4, train_data=train_data_combine, threshold= 0.5))

In [ ]:
model_4_results_df= pd.concat( results_list, ignore_index = True)

In [ ]:
results_list = []

results_list.append( fit_and_assess_logistic('model_5', model_5, train_data=train_data_combine, threshold= 0.5))

In [ ]:
model_5_results_df= pd.concat( results_list, ignore_index = True)

In [ ]:
results_list = []

results_list.append( fit_and_assess_logistic('model_6', model_6, train_data=train_data_combine, threshold= 0.5))

In [ ]:
model_6_results_df= pd.concat( results_list, ignore_index = True)

In [ ]:
results_list = []

results_list.append( fit_and_assess_logistic('model_7', model_7, train_data=train_data_combine, threshold= 0.5))

In [ ]:
model_7_results_df= pd.concat( results_list, ignore_index = True)

In [ ]:
results_list = []

results_list.append( fit_and_assess_logistic('model_8', model_8, train_data=train_data_combine, threshold= 0.5))

In [ ]:
model_8_results_df= pd.concat( results_list, ignore_index = True)

In [ ]:
results_list = []

results_list.append( fit_and_assess_logistic('model_9', model_9, train_data=train_data_combine, threshold= 0.5))

In [ ]:
model_9_results_df= pd.concat( results_list, ignore_index = True)

In [ ]:
results_list = []

results_list.append( fit_and_assess_logistic('model_10', model_10, train_data=train_data_combine, threshold= 0.5))

In [ ]:
model_10_results_df= pd.concat( results_list, ignore_index = True)

In [ ]:
results_list = []

results_list.append( fit_and_assess_logistic('model_11', model_11, train_data=train_data_combine, threshold= 0.5))

In [ ]:
model_11_results_df= pd.concat( results_list, ignore_index = True)

In [ ]:
results_list = []

results_list.append( fit_and_assess_logistic('model_12', model_12, train_data=train_data_combine, threshold= 0.5))

In [ ]:
model_12_results_df= pd.concat( results_list, ignore_index = True)

To access coeffients and other performance metrics, I will need to fit on training set again

In [ ]:
model_1_fit = smf.logit(formula= model_1 , data = train_data_combine).fit()

In [ ]:
model_2_fit = smf.logit(formula= model_2 , data = train_data_combine).fit()

In [ ]:
model_3_fit = smf.logit(formula= model_3 , data = train_data_combine).fit()

In [ ]:
model_4_fit = smf.logit(formula= model_4 , data = train_data_combine).fit()

In [ ]:
model_5_fit = smf.logit(formula= model_5 , data = train_data_combine).fit()

In [ ]:
model_6_fit = smf.logit(formula= model_6 , data = train_data_combine).fit()

In [ ]:
model_7_fit = smf.logit(formula= model_7 , data = train_data_combine).fit()

In [ ]:
model_8_fit = smf.logit(formula= model_8 , data = train_data_combine).fit()

In [ ]:
model_9_fit = smf.logit(formula= model_9 , data = train_data_combine).fit()

In [ ]:
model_10_fit = smf.logit(formula= model_10 , data = train_data_combine).fit()

In [ ]:
model_11_fit = smf.logit(formula= model_11 , data = train_data_combine).fit()

In [ ]:
model_12_fit = smf.logit(formula= model_12 , data = train_data_combine).fit()

In [ ]:
model_1_results_df

In [ ]:
my_coefplot(model_1_fit)

In [ ]:
model_2_results_df

In [ ]:
my_coefplot(model_2_fit)

In [ ]:
model_3_results_df

In [ ]:
my_coefplot(model_3_fit)

In [ ]:
model_4_results_df

In [ ]:
my_coefplot(model_4_fit)

In [ ]:
model_5_results_df

In [ ]:
my_coefplot(model_5_fit)

In [ ]:
model_6_results_df

In [ ]:
my_coefplot(model_6_fit)

In [ ]:
model_7_results_df


In [ ]:
my_coefplot(model_7_fit)

In [ ]:
model_8_results_df

In [ ]:
my_coefplot(model_8_fit)

In [ ]:
model_9_results_df

In [ ]:
my_coefplot(model_9_fit)

In [ ]:
model_10_results_df

In [ ]:
my_coefplot(model_10_fit)

In [ ]:
model_11_results_df

In [ ]:
my_coefplot(model_11_fit)

In [ ]:
model_12_results_df

In [ ]:
my_coefplot(model_12_fit)

#### D. Models: Conclusion


All performance metrics are displaye above. The my_coefplot visualiation displays the coeficents that are statsitically significant, as these do not overlap 0. The magnitude can be seen as the distance of the coefficient from the intercept, additional notation would be redudant. Below I will repeat this detail this for top performing model using performance metrics

***BEST PERFORMING MODEL***

The Best performing model is model_10. 

Model_10 has 432 features.

The accuracy is 0.697235 which is better than the intercept only model (0.612726)

The feature with greatest magnitude is `energy:playlist_subgenre[T.hip pop]` at 6.674204

The feature that is most significally significat is `playlist_subgenre[T.dance pop]` with pvalue of 0.00002





In [ ]:
model_10_fit.params.abs().sort_values(ascending=False).head(10)

In [ ]:
model_10_fit.pvalues[model_10_fit.pvalues <= 0.05].sort_values(ascending=True)

In [ ]:
model_10_fit.bse

---

## E.	Models: Predictions

#### Test Data Creation

Based on the model_10 coeffifient performance, I will focus on the below 3 inputs as the most important. 

1. (Categorical) `playlist_subgenre` (all unique values)
2. (Continuous) `energy` (101 unique values)
3. (Continuous) `loudness` (5 unique values)
4. All other inputs = CONSTANT, Median for continuous and most frequent value for Categorical



In [ ]:
predict_df = train_data_combine.head(101).copy()

In [ ]:
predict_df.info()

In [ ]:
train_data_combine.nunique()

#### Creating the focused inputs

In [ ]:
import math

In [ ]:
playlist_subgenre_array = np.tile(train_data_combine['playlist_subgenre'].unique(), math.ceil(101 / 24))

In [ ]:
playlist_subgenre_101 = playlist_subgenre_array[:101]

In [ ]:
len(playlist_subgenre_101)

In [ ]:
loudness_array = np.linspace(train_data_combine.loudness.min(), train_data_combine.loudness.max(), num= 5)

In [ ]:
loudness_array

In [ ]:
loudness_array_105 = np.tile(loudness_array, math.ceil(101 / 5))

In [ ]:
loudness_array_101 = loudness_array_105[:101]

In [ ]:
loudness_array_101

In [ ]:
predict_df['energy'] = np.linspace(train_data_combine.energy.min(), train_data_combine.energy.max(), num= 101)

In [ ]:
predict_df['loudness'] = loudness_array_101

In [ ]:
predict_df['playlist_subgenre'] = playlist_subgenre_101

#### Creating the remaining categorical inputs

In [ ]:
predict_df['playlist_genre'] = train_data_combine['playlist_genre'].mode().iloc[0]

In [ ]:
predict_df['binary_outcome'] = train_data_combine['binary_outcome'].mode().iloc[0]

In [ ]:
predict_df['mode'] = train_data_combine['mode'].mode().iloc[0]

#### Creating the remaining continuous inputs

In [ ]:
predict_df['valence'] = train_data_combine['valence'].mean()

In [ ]:
predict_df['count'] = train_data_combine['count'].mean()

In [ ]:
predict_df['liveness'] = train_data_combine['liveness'].mean()

In [ ]:
predict_df['danceability'] = train_data_combine['danceability'].mean()

In [ ]:
predict_df['speechiness'] = train_data_combine['speechiness'].mean()

In [ ]:
predict_df['acousticness'] = train_data_combine['acousticness'].mean()

In [ ]:
predict_df['instrumentalness'] = train_data_combine['instrumentalness'].mean()

In [ ]:
predict_df['duration_ms'] = train_data_combine['duration_ms'].mean()

In [ ]:
predict_df['release_year'] = train_data_combine['release_year'].mean()

In [ ]:
predict_df['tempo'] = train_data_combine['tempo'].mean()

In [ ]:
predict_df['key'] = train_data_combine['key'].mean()

In [ ]:
predict_df.nunique()

In [ ]:
dfviz = predict_df.copy()

In [ ]:
dfviz['pred_probability_mod_10'] = model_10_fit.predict (predict_df)

In [ ]:
fig, ax = plt.subplots(figsize= (12,6))

sns.scatterplot( data = train_data_combine, x = 'playlist_subgenre', y = 'binary_outcome', s = 150, alpha = 0.5)

ax.axhline( y = 0.25, color = 'green', linestyle = '--', linewidth = 2)

ax.axhline( y = 0.5, color = 'grey', linestyle = '--', linewidth = 2)

ax.axhline( y = 0.75, color = 'orange', linestyle = '--', linewidth = 2)

ax.axvline( x = -1, color = 'black', linestyle = ':')

ax.axvline( x = 1, color = 'black', linestyle = ':')

sns.lineplot(data = dfviz, x = 'playlist_subgenre', y = 'pred_probability_mod_10', color = 'k')

plt.show()

In [ ]:
dfviz['pred_probability_mod_8'] = model_8_fit.predict (predict_df)

In [ ]:
fig, ax = plt.subplots(figsize= (12,6))

sns.scatterplot( data = train_data_combine, x = 'playlist_subgenre', y = 'binary_outcome', s = 150, alpha = 0.5)

ax.axhline( y = 0.25, color = 'green', linestyle = '--', linewidth = 2)

ax.axhline( y = 0.5, color = 'grey', linestyle = '--', linewidth = 2)

ax.axhline( y = 0.75, color = 'orange', linestyle = '--', linewidth = 2)

ax.axvline( x = -1, color = 'black', linestyle = ':')

ax.axvline( x = 1, color = 'black', linestyle = ':')

sns.lineplot(data = dfviz, x = 'playlist_subgenre', y = 'pred_probability_mod_8', color = 'k')

plt.show()

Inference: The visualisions did not render as clean as the sample during course, but I can see the spikes are above 50% for average accuracy

---

## F.	Models: Performance and Validation (25 points)

Creating training dataset

In [ ]:
from sklearn.model_selection import StratifiedKFold

In [ ]:
kf = StratifiedKFold (n_splits=5, shuffle=True, random_state=9483156)

In [ ]:
import statsmodels.formula.api as smf

In [ ]:
formula_list = ['binary_outcome ~ mode + playlist_genre + playlist_subgenre + key + mode',
                #model_6
                'binary_outcome ~ danceability + energy+ loudness + speechiness + acousticness+ instrumentalness + liveness + valence + tempo + duration_ms + release_year + playlist_genre + playlist_subgenre + key + mode + count',
                #model_8
                'binary_outcome ~ (danceability + energy+ loudness + speechiness + acousticness + instrumentalness + liveness + tempo) * playlist_subgenre * key']
                #model_10


In [ ]:
input_names = df_clean.drop(columns=['binary_outcome']).copy().columns.to_list()

In [ ]:
output_name='binary_outcome'

In [ ]:
def train_and_test_logistic_with_cv(mod_name, a_formula, data_df, x_names, y_name, cv, threshold=.5):
    # separate the inputs and output
    input_df = data_df.loc[ :, x_names ].copy()

    # initiate the perforamcne metric storage lists
    # will include train_res for teaching
    train_res = []
    test_res = []

    # SPLIT the data and iterate over the folds
    for train_id, test_id in cv.split (input_df.to_numpy(), data_df[ y_name ].to_numpy() ):
        # subset the training and test splits within each fold
        train_data = data_df.iloc[ train_id, :].copy()
        test_data = data_df.iloc [ test_id, :].copy()

        # FIT the model on the TRAINING data within the current FOLD
        a_mod = smf.logit(formula=a_formula, data=train_data).fit()

        #PREDICT the TRAINING set within each FOLD (for teaching purposes)
        train_copy = train_data.copy()
        train_copy['pred_probability'] = a_mod.predict( train_data)
        train_copy['pred_class'] = np.where ( train_copy.pred_probability > threshold, 1, 0)

        #PREDICT the TESTING within each FOLD
        test_copy = test_data.copy()
        test_copy['pred_probability'] = a_mod.predict( test_data)
        test_copy['pred_class'] = np.where ( test_copy.pred_probability > threshold, 1, 0)

        # calculate the PERFORMANCE METRIC on the TRAINING SET within the FOLD
        train_res.append(np.mean(train_copy[y_name] == train_copy.pred_class))

        # calculate the PERFORMNACE METRIC on the TESTING SET within the FOLD
        test_res.append(np.mean(test_copy[y_name] == test_copy.pred_class))

    # book keeping to store the results
    train_df = pd.DataFrame({'Accuracy': train_res})
    train_df['from_set'] = 'training'
    train_df['fold_id'] = train_df.index + 1

    # book keeping to store the results
    test_df = pd.DataFrame({'Accuracy': test_res})
    test_df['from_set'] = 'testing'
    test_df['fold_id'] = test_df.index + 1

    #combine the splits together
    res_df = pd.concat([train_df, test_df], ignore_index=True)

    res_df['model_name'] = mod_name
    res_df['model_formula'] = a_formula
    res_df['num_coefs'] = len( a_mod.params)
    res_df['threshold'] = threshold

    return res_df
        

In [ ]:
train_and_test_logistic_with_cv(0,formula_list[0], data_df=df_clean, x_names=input_names, y_name=output_name, cv=kf)

In [ ]:
train_and_test_logistic_with_cv(2,formula_list[2], data_df=df_clean, x_names=input_names, y_name=output_name, cv=kf)

In [ ]:
train_and_test_logistic_with_cv(1,formula_list[1], data_df=df_clean, x_names=input_names, y_name=output_name, cv=kf)

In [ ]:
results_list=[]

for m in range (len(formula_list) ):
    print('-- Formula ID %d --' % m )

    results_list.append ( train_and_test_logistic_with_cv(m, formula_list[m], data_df = df_clean, x_names = input_names, y_name=output_name, cv=kf))

In [ ]:
cv_results = pd.concat(results_list, ignore_index=True)

In [ ]:
cv_results

In [ ]:
cv_results.model_name.value_counts()

In [ ]:
sns.catplot ( data= cv_results, x ='model_name', hue='from_set', kind='count')

plt.show()

In [ ]:
sns.catplot(data = cv_results, x='model_name', y='Accuracy', hue='from_set')

plt.show()

The strip plot shows that `model_10`, the most complex, has the highest accuracy for the training data, buthard to tell the best average

In [ ]:
fig, ax = plt.subplots()

sns.stripplot(data = cv_results, x='model_name', y='Accuracy', hue ='from_set', alpha=0.5, ax=ax)

sns.pointplot(data=cv_results, x='model_name', y='Accuracy',linestyle= 'none', join=False, ax=ax)

plt.show()


In [ ]:
fig, ax = plt.subplots()

sns.pointplot(data=cv_results, x='model_name', y='Accuracy', hue ='from_set', linestyle= 'none',  ax=ax)

plt.show()

In [ ]:
fig, ax = plt.subplots()

sns.pointplot(data=cv_results.loc[cv_results.from_set == 'testing', :], x='model_name', y='Accuracy', hue ='from_set', linestyle= 'none',  ax=ax)

plt.show()

From above plots, we see `model_8` and `model_10` have similiar performnace, but `model_10` slightly better. 

In [ ]:
fig, ax = plt.subplots()

sns.pointplot(data=cv_results.loc[cv_results.from_set == 'testing', :], 
              x='model_name', y='Accuracy', hue ='from_set', linestyle= 'none',
              errorbar=('ci', 68) )

plt.show()

with 68% confidence intraval, `model_10` is still the best... But has 10x the number of coefficents than `model_8` (see below). 

the differnce in model_8 and model_10 is small, and I think the added complexity could create overfitting problems with new real-world date. Therefore, I will use `model_8` as the BEST model to use 

In [ ]:
cv_results

In [ ]:
sns.catplot(data=cv_results.loc[cv_results.from_set == 'testing', :], 
           x ='num_coefs', y='Accuracy', kind='point', linestyle='none',
           hue='model_name',
           errorbar=('ci', 68),
           aspect=1)
plt.show()



## Model selection 
choose the model that does BEST **on average** on NEW data

we must FIT the model one more time on ENTIRE data set

In [ ]:
best_model = smf.logit(formula= formula_list[1], data = df_clean).fit()

In [ ]:
best_model.pvalues < 0.05

In [ ]:
best_model.params

In [ ]:
np.abs( best_model.params [ best_model.pvalues < 0.05]).sort_values(ascending = False)

### Conclusion

We used statisfied kfolds to create 5 folds of cross validation. We trained 3 models on the test data, then applied the same models on the training data. We see both model_8 and model_10 have similiar accuracy, but model_8 is much much simpler than model_10 and could reduce overfitting on new data. 

We deem model_8 as the model to choose. 

#### Comparing  Pipeline with elasticnet

In [ ]:
xinputs_num = df_clean[['key', 'mode', 'danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'release_year', 'count']].copy()

In [ ]:
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

In [ ]:
xinputs_cat = df_clean[['playlist_genre', 'playlist_subgenre']].copy()

In [ ]:
X_encoded_array = encoder.fit_transform(xinputs_cat)

In [ ]:
new_col_names = encoder.get_feature_names_out(['playlist_genre', 'playlist_subgenre'])

In [ ]:
new_col_names

In [ ]:
df_encoded = pd.DataFrame(X_encoded_array, columns=new_col_names, index=df_clean.index)

In [ ]:
df_final = pd.concat([xinputs_num, df_encoded], axis=1)

In [ ]:
youtput= df_clean.loc[:, ['binary_outcome']].copy()

In [ ]:
youtput.rename(columns={'binary_outcome': 'response'}, inplace=True)

In [ ]:
X_train = df_final.to_numpy()

In [ ]:
y_train = youtput.response.to_numpy().ravel()

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
X_train.shape

In [ ]:
from sklearn.pipeline import Pipeline

In [ ]:
enet_to_fit = LogisticRegression(penalty='elasticnet', solver='saga', random_state=202, max_iter=25001, fit_intercept=True)

In [ ]:
from sklearn.compose import ColumnTransformer

In [ ]:
from sklearn.preprocessing import OneHotEncoder

In [ ]:
enet_wflow=Pipeline(steps=[('standarize', StandardScaler()), 
                           ('enet', enet_to_fit)] )

In [ ]:
enet_wflow

In [ ]:
from sklearn.model_selection import StratifiedKFold

In [ ]:
kf= StratifiedKFold (n_splits=5, shuffle=True, random_state=101)

In [ ]:
kf.get_n_splits()

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
enet_grid = {'enet__C': np.exp( np.linspace(-10, 10, num=17) ),
             'enet__l1_ratio': np.linspace(0, 1, num=3)}

In [ ]:
enet_search = GridSearchCV(enet_wflow, param_grid=enet_grid, cv=kf)

In [ ]:
enet_search_resutls = enet_search.fit( X_train, y_train) 

In [ ]:
enet_search_resutls.best_params_

The optimal l1 ratio is  '0.0'. this means the elastinet converted to ridge. This did not happen, so it  is indicating the input coorelation is NOT causing problems.  

In [ ]:
enet_search_resutls.best_score_

The best score of 68.44 is very close to the best score of a few models ran during fitting, such as `model_8` and `model_10` which were both 68 -69 accuracy and better than intercept only. 



In [ ]:
enet_search_resutls.best_estimator_

In [ ]:
enet_search_resutls.best_estimator_.named_steps['enet'].coef_

In [ ]:
enet_search_resutls.classes_